<a href="https://colab.research.google.com/github/attabeezy/crop-guard/blob/main/notebooks/04_evaluate_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CropGuard — held-out evaluation, calibration, and Grad-CAM

This notebook enforces the evaluation boundary:

1. Compare frozen and fine-tuned checkpoints on **validation macro-F1**.
2. Calibrate confidence and lock the abstention threshold using validation only.
3. Load the held-out test set after those decisions are fixed.
4. Evaluate the test set once and generate failure/Grad-CAM evidence.

Do not change the model or threshold in response to test results.

## 1. Runtime and dependencies

Select a T4 GPU runtime before continuing.

In [ ]:
%pip install -q kagglehub pandas scikit-learn scipy seaborn pillow tqdm

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 2026
IMAGE_SIZE = 224
BATCH_SIZE = 32
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("No GPU detected. Select a T4 GPU runtime.")
print("TensorFlow:", tf.__version__)
print("GPU:", gpus[0])

## 2. Download and verify pinned inputs

The manifests come from the immutable audit commit. Model binaries are fetched from Git LFS through GitHub's media endpoint and checked byte-for-byte.

In [ ]:
import hashlib
import json
import shutil
import urllib.request
import zipfile

AUDIT_COMMIT = "4c8b366df4c251d6424258c3cb74a58f2289ee16"
AUDIT_SHA256 = "1f69013b0a0fc58cba5dccab37d50e8767902580f443ac13daa3fba885e3269c"
MODEL_COMMIT = "08d86ed7807075b195a43acb80e11ef738ee4b28"
MODEL_HASHES = {
    "best_frozen.keras": "f9c1b9b2fd2278c0491cf23cb5f8abe7564e225002b75370474bd11eb9f5b0dd",
    "best_finetuned.keras": "14c7ed601c3bebd64238a1a3375c1653bc6017009df645a22878241525e0aa9d",
}

input_dir = Path("/content/cropguard_evaluation_inputs")
if input_dir.exists():
    shutil.rmtree(input_dir)
input_dir.mkdir()

audit_zip = input_dir / "cropguard-data-prep.zip"
audit_url = (
    "https://raw.githubusercontent.com/attabeezy/crop-guard/"
    f"{AUDIT_COMMIT}/cropguard-data-prep.zip"
)
urllib.request.urlretrieve(audit_url, audit_zip)
if hashlib.sha256(audit_zip.read_bytes()).hexdigest() != AUDIT_SHA256:
    raise RuntimeError("Audit ZIP hash mismatch")
audit_dir = input_dir / "audit"
with zipfile.ZipFile(audit_zip) as archive:
    archive.extractall(audit_dir)

for filename, expected_hash in MODEL_HASHES.items():
    url = (
        "https://media.githubusercontent.com/media/attabeezy/crop-guard/"
        f"{MODEL_COMMIT}/results/training-artifacts/{filename}"
    )
    destination = input_dir / filename
    urllib.request.urlretrieve(url, destination)
    actual_hash = hashlib.sha256(destination.read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f"Model hash mismatch for {filename}: {actual_hash}")
    print(f"Verified {filename}: {destination.stat().st_size / 1_000_000:.1f} MB")

config_url = (
    "https://raw.githubusercontent.com/attabeezy/crop-guard/"
    f"{MODEL_COMMIT}/results/training-artifacts/training_config.json"
)
with urllib.request.urlopen(config_url) as response:
    training_config = json.loads(response.read().decode("utf-8"))
classes = training_config["class_order"]
assert len(classes) == 22
display(training_config)

## 3. Download the same CCMT mirror and prepare validation

The test manifest is deliberately not loaded in this section.

In [ ]:
import kagglehub

DATASET_HANDLE = "nirmalsankalana/crop-pest-and-disease-detection"
dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
class_to_index = {label: index for index, label in enumerate(classes)}

def resolve_manifest(filename):
    frame = pd.read_csv(audit_dir / filename)
    frame["absolute_path"] = frame["path"].map(lambda value: str(dataset_root / value))
    frame["target"] = frame["label"].map(class_to_index)
    if frame["target"].isna().any():
        raise RuntimeError(f"Unknown label in {filename}")
    missing = frame[~frame["absolute_path"].map(lambda value: Path(value).is_file())]
    if len(missing):
        raise RuntimeError(f"{len(missing)} files missing for {filename}")
    return frame

validation_frame = resolve_manifest("validation.csv")
print(f"Validation images: {len(validation_frame):,}")

from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def image_integrity_check(row):
    try:
        raw = Path(row.absolute_path).read_bytes()
        actual_hash = hashlib.sha256(raw).hexdigest()
        if actual_hash != row.sha256:
            return {"kind": "hash_mismatch", "path": row.path, "label": row.label,
                    "error": f"expected {row.sha256}; got {actual_hash}"}
        image = tf.io.decode_image(
            tf.convert_to_tensor(raw), channels=3, expand_animations=False
        )
        _ = image.numpy()
        return None
    except Exception as exc:
        return {"kind": "decode_failure", "path": row.path, "label": row.label,
                "error": f"{type(exc).__name__}: {exc}"}

def verify_and_filter(frame, name):
    rows = list(frame.itertuples(index=False))
    with ThreadPoolExecutor(max_workers=8) as pool:
        checked = list(tqdm(pool.map(image_integrity_check, rows), total=len(rows), desc=name))
    failures = pd.DataFrame([item for item in checked if item is not None])
    if len(failures) and (failures["kind"] == "hash_mismatch").any():
        raise RuntimeError(f"{name} contains image hash mismatches; dataset version changed")
    rejected = set(failures["path"]) if len(failures) else set()
    filtered = frame[~frame["path"].isin(rejected)].reset_index(drop=True)
    print(f"{name}: verified {len(frame):,}; decode exclusions {len(rejected):,}")
    return filtered, failures

validation_frame, validation_decode_failures = verify_and_filter(
    validation_frame, "validation"
)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path, target):
    image = tf.io.decode_image(
        tf.io.read_file(path), channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
    return image, target

def make_dataset(frame):
    dataset = tf.data.Dataset.from_tensor_slices((
        frame["absolute_path"].to_numpy(),
        frame["target"].to_numpy(dtype=np.int32),
    ))
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

validation_dataset = make_dataset(validation_frame)

## 4. Select the checkpoint using validation macro-F1

Macro-F1 weights every class equally and is the primary selection metric because class sizes vary by roughly 13×.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, log_loss

validation_truth = validation_frame["target"].to_numpy(dtype=np.int32)
models = {}
validation_probabilities = {}
selection_rows = []

for filename in MODEL_HASHES:
    model = tf.keras.models.load_model(input_dir / filename, compile=False)
    if model.output_shape[-1] != len(classes):
        raise RuntimeError(f"Output mismatch for {filename}: {model.output_shape}")
    probabilities = model.predict(validation_dataset, verbose=1)
    predictions = probabilities.argmax(axis=1)
    row = {
        "checkpoint": filename,
        "validation_accuracy": accuracy_score(validation_truth, predictions),
        "validation_macro_f1": f1_score(
            validation_truth, predictions, average="macro"
        ),
        "validation_log_loss": log_loss(
            validation_truth, probabilities, labels=np.arange(len(classes))
        ),
    }
    models[filename] = model
    validation_probabilities[filename] = probabilities
    selection_rows.append(row)

checkpoint_selection = pd.DataFrame(selection_rows).sort_values(
    ["validation_macro_f1", "validation_log_loss"], ascending=[False, True]
).reset_index(drop=True)
display(checkpoint_selection)
selected_name = checkpoint_selection.iloc[0]["checkpoint"]
selected_model = models[selected_name]
validation_probs_raw = validation_probabilities[selected_name]
print("LOCKED CHECKPOINT:", selected_name)

## 5. Calibrate confidence on validation only

Temperature scaling adjusts confidence without changing the predicted class. Expected calibration error (ECE) measures agreement between confidence and observed accuracy.

In [ ]:
from scipy.optimize import minimize_scalar

def temperature_scale(probabilities, temperature):
    logits = np.log(np.clip(probabilities, 1e-8, 1.0)) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    exponentiated = np.exp(logits)
    return exponentiated / exponentiated.sum(axis=1, keepdims=True)

def expected_calibration_error(truth, probabilities, bins=15):
    confidence = probabilities.max(axis=1)
    correct = probabilities.argmax(axis=1) == truth
    total = len(truth)
    ece = 0.0
    edges = np.linspace(0.0, 1.0, bins + 1)
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.any():
            ece += mask.sum() / total * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)

def validation_nll(temperature):
    calibrated = temperature_scale(validation_probs_raw, temperature)
    return log_loss(validation_truth, calibrated, labels=np.arange(len(classes)))

optimization = minimize_scalar(validation_nll, bounds=(0.25, 5.0), method="bounded")
temperature = float(optimization.x)
validation_probs = temperature_scale(validation_probs_raw, temperature)
calibration_summary = {
    "temperature": temperature,
    "validation_nll_before": log_loss(validation_truth, validation_probs_raw),
    "validation_nll_after": log_loss(validation_truth, validation_probs),
    "validation_ece_before": expected_calibration_error(
        validation_truth, validation_probs_raw
    ),
    "validation_ece_after": expected_calibration_error(
        validation_truth, validation_probs
    ),
}
display(calibration_summary)

## 6. Lock the uncertainty threshold

Policy: choose the lowest threshold that reaches at least 90% accepted-case validation accuracy while retaining at least 30% coverage. If none qualifies, use the threshold with the best accepted accuracy among thresholds retaining 30% coverage.

In [ ]:
TARGET_ACCEPTED_ACCURACY = 0.90
MINIMUM_COVERAGE = 0.30
validation_confidence = validation_probs.max(axis=1)
validation_predictions = validation_probs.argmax(axis=1)
validation_correct = validation_predictions == validation_truth
policy_rows = []

for threshold in np.round(np.arange(0.40, 0.951, 0.01), 2):
    accepted = validation_confidence >= threshold
    policy_rows.append({
        "threshold": float(threshold),
        "coverage": float(accepted.mean()),
        "accepted_count": int(accepted.sum()),
        "accepted_accuracy": float(validation_correct[accepted].mean())
            if accepted.any() else np.nan,
    })

confidence_policy = pd.DataFrame(policy_rows)
eligible = confidence_policy[
    (confidence_policy["coverage"] >= MINIMUM_COVERAGE)
    & (confidence_policy["accepted_accuracy"] >= TARGET_ACCEPTED_ACCURACY)
]
if len(eligible):
    threshold_row = eligible.sort_values("threshold").iloc[0]
    threshold_rule = "lowest threshold meeting accuracy and coverage targets"
else:
    candidates = confidence_policy[confidence_policy["coverage"] >= MINIMUM_COVERAGE]
    threshold_row = candidates.sort_values(
        ["accepted_accuracy", "coverage"], ascending=[False, False]
    ).iloc[0]
    threshold_rule = "fallback: best accepted accuracy with minimum coverage"

confidence_threshold = float(threshold_row["threshold"])
print("LOCKED TEMPERATURE:", temperature)
print("LOCKED CONFIDENCE THRESHOLD:", confidence_threshold)
print("Rule:", threshold_rule)
display(threshold_row.to_frame().T)

# Evaluation boundary

The checkpoint, temperature, and threshold are now locked. The following cells load the held-out test manifest for the first and only evaluation.

## 7. Load and TensorFlow-validate the held-out test set

Unreadable files are excluded solely for technical decodability and reported explicitly.

In [ ]:
test_frame = resolve_manifest("test.csv")
test_frame, test_decode_failures = verify_and_filter(test_frame, "test")
print(f"Test images evaluated: {len(test_frame):,}")
print(f"Test decode exclusions: {len(test_decode_failures):,}")
test_dataset = make_dataset(test_frame)

## 8. Run the single held-out evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_truth = test_frame["target"].to_numpy(dtype=np.int32)
test_probs_raw = selected_model.predict(test_dataset, verbose=1)
test_probs = temperature_scale(test_probs_raw, temperature)
test_predictions = test_probs.argmax(axis=1)
test_confidence = test_probs.max(axis=1)
test_accepted = test_confidence >= confidence_threshold
test_correct = test_predictions == test_truth

report = classification_report(
    test_truth,
    test_predictions,
    labels=np.arange(len(classes)),
    target_names=classes,
    output_dict=True,
    zero_division=0,
)
per_class_metrics = pd.DataFrame(report).T
test_summary = {
    "checkpoint": selected_name,
    "evaluated_images": len(test_frame),
    "decode_exclusions": len(test_decode_failures),
    "accuracy": float(accuracy_score(test_truth, test_predictions)),
    "macro_f1": float(f1_score(test_truth, test_predictions, average="macro")),
    "weighted_f1": float(f1_score(test_truth, test_predictions, average="weighted")),
    "log_loss_calibrated": float(log_loss(test_truth, test_probs)),
    "ece_calibrated": expected_calibration_error(test_truth, test_probs),
    "temperature_locked_on_validation": temperature,
    "confidence_threshold_locked_on_validation": confidence_threshold,
    "coverage": float(test_accepted.mean()),
    "accepted_accuracy": float(test_correct[test_accepted].mean())
        if test_accepted.any() else None,
    "abstained_images": int((~test_accepted).sum()),
}
display(test_summary)
display(per_class_metrics)

## 9. Confusion matrices and healthy-class performance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

confusion = confusion_matrix(test_truth, test_predictions, labels=np.arange(len(classes)))
row_totals = confusion.sum(axis=1, keepdims=True)
normalized_confusion = np.divide(
    confusion, row_totals, out=np.zeros_like(confusion, dtype=float), where=row_totals != 0
)

fig, axes = plt.subplots(1, 2, figsize=(28, 12))
sns.heatmap(confusion, cmap="Blues", ax=axes[0], xticklabels=classes,
            yticklabels=classes, cbar=False)
axes[0].set_title("Held-out test confusion matrix — counts")
sns.heatmap(normalized_confusion, cmap="Blues", ax=axes[1], xticklabels=classes,
            yticklabels=classes, vmin=0, vmax=1)
axes[1].set_title("Held-out test confusion matrix — row normalized")
for axis in axes:
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
    axis.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

healthy_labels = [label for label in classes if label.endswith("|healthy")]
healthy_metrics = per_class_metrics.loc[healthy_labels, ["precision", "recall", "f1-score", "support"]]
display(healthy_metrics)

## 10. Confidence policy and failure gallery

In [ ]:
test_predictions_frame = test_frame[["path", "crop", "class", "label"]].copy()
test_predictions_frame["actual_index"] = test_truth
test_predictions_frame["predicted_index"] = test_predictions
test_predictions_frame["predicted_label"] = [classes[index] for index in test_predictions]
test_predictions_frame["raw_confidence"] = test_probs_raw.max(axis=1)
test_predictions_frame["calibrated_confidence"] = test_confidence
test_predictions_frame["correct"] = test_correct
test_predictions_frame["accepted"] = test_accepted

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(confidence_policy["threshold"], confidence_policy["coverage"])
axes[0].axvline(confidence_threshold, color="red", linestyle="--")
axes[0].set_title("Validation coverage")
axes[0].set_xlabel("Confidence threshold")
axes[0].set_ylabel("Coverage")
axes[1].plot(confidence_policy["threshold"], confidence_policy["accepted_accuracy"])
axes[1].axvline(confidence_threshold, color="red", linestyle="--")
axes[1].axhline(TARGET_ACCEPTED_ACCURACY, color="gray", linestyle=":")
axes[1].set_title("Validation accepted-case accuracy")
axes[1].set_xlabel("Confidence threshold")
axes[1].set_ylabel("Accuracy")
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

from PIL import Image, ImageOps
failure_indices = np.flatnonzero(~test_correct)
rng = np.random.default_rng(SEED)
shown_failures = rng.choice(failure_indices, size=min(12, len(failure_indices)), replace=False)
fig, axes = plt.subplots(3, 4, figsize=(16, 13))
for axis, index in zip(axes.flat, shown_failures):
    with Image.open(test_frame.iloc[index]["absolute_path"]) as source:
        axis.imshow(ImageOps.exif_transpose(source).convert("RGB"))
    axis.set_title(
        f"Actual: {classes[test_truth[index]]}\n"
        f"Pred: {classes[test_predictions[index]]}\n"
        f"Confidence: {test_confidence[index]:.1%}", fontsize=8
    )
    axis.axis("off")
for axis in axes.flat[len(shown_failures):]:
    axis.axis("off")
plt.tight_layout()
failure_figure = fig
plt.show()

## 11. Grad-CAM honesty check

Heatmaps target the predicted class. They are evidence for review, not proof that the diagnosis is correct.

In [ ]:
backbone = next(
    layer for layer in selected_model.layers
    if isinstance(layer, tf.keras.Model) and layer.name.startswith("efficientnet")
)
last_conv = backbone.get_layer("top_conv")
feature_extractor = tf.keras.Model(
    backbone.input, [last_conv.output, backbone.output], name="gradcam_features"
)
gradcam_input = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
augmented = selected_model.get_layer("training_augmentation")(
    gradcam_input, training=False
)
conv_features, backbone_output = feature_extractor(augmented, training=False)
pooled = selected_model.get_layer("global_average_pool")(backbone_output)
dropped = selected_model.get_layer("classifier_dropout")(pooled, training=False)
gradcam_predictions = selected_model.get_layer("probabilities")(dropped)
gradcam_model = tf.keras.Model(
    gradcam_input, [conv_features, gradcam_predictions]
)

def load_model_image(path):
    image, _ = decode_image(tf.constant(path), tf.constant(0))
    return tf.expand_dims(image, axis=0)

def make_gradcam(path, class_index):
    image = load_model_image(path)
    with tf.GradientTape() as tape:
        features, predictions = gradcam_model(image, training=False)
        score = predictions[:, class_index]
    gradients = tape.gradient(score, features)
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.reduce_sum(weights * features, axis=-1)[0]
    heatmap = tf.maximum(heatmap, 0)
    maximum = tf.reduce_max(heatmap)
    heatmap = tf.where(maximum > 0, heatmap / maximum, heatmap)
    return heatmap.numpy()

correct_indices = np.flatnonzero(test_correct)
incorrect_indices = np.flatnonzero(~test_correct)
review_indices = np.concatenate([
    rng.choice(correct_indices, size=min(4, len(correct_indices)), replace=False),
    rng.choice(incorrect_indices, size=min(4, len(incorrect_indices)), replace=False),
])

fig, axes = plt.subplots(2, 4, figsize=(17, 9))
for axis, index in zip(axes.flat, review_indices):
    path = test_frame.iloc[index]["absolute_path"]
    with Image.open(path) as source:
        original = np.asarray(ImageOps.exif_transpose(source).convert("RGB"))
    heatmap = make_gradcam(path, int(test_predictions[index]))
    axis.imshow(original)
    axis.imshow(heatmap, cmap="jet", alpha=0.42, extent=(0, original.shape[1], original.shape[0], 0))
    status = "correct" if test_correct[index] else "WRONG"
    axis.set_title(
        f"{status}: {classes[test_predictions[index]]}\n"
        f"actual: {classes[test_truth[index]]}", fontsize=8
    )
    axis.axis("off")
plt.tight_layout()
gradcam_figure = fig
plt.show()

## 12. Export evaluation artifacts

In [ ]:
from datetime import datetime, timezone

output_dir = Path("/content/cropguard_evaluation_results")
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir()

checkpoint_selection.to_csv(output_dir / "checkpoint_selection.csv", index=False)
per_class_metrics.to_csv(output_dir / "per_class_metrics.csv")
healthy_metrics.to_csv(output_dir / "healthy_class_metrics.csv")
pd.DataFrame(confusion, index=classes, columns=classes).to_csv(
    output_dir / "confusion_matrix.csv"
)
pd.DataFrame(normalized_confusion, index=classes, columns=classes).to_csv(
    output_dir / "confusion_matrix_normalized.csv"
)
confidence_policy.to_csv(output_dir / "validation_confidence_policy.csv", index=False)
test_predictions_frame.to_csv(output_dir / "test_predictions.csv", index=False)
validation_decode_failures.to_csv(
    output_dir / "validation_decode_failures.csv", index=False
)
test_decode_failures.to_csv(output_dir / "test_decode_failures.csv", index=False)
failure_figure.savefig(output_dir / "failure_gallery.png", dpi=180, bbox_inches="tight")
gradcam_figure.savefig(output_dir / "gradcam_gallery.png", dpi=180, bbox_inches="tight")

evaluation_summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_commit": MODEL_COMMIT,
    "audit_commit": AUDIT_COMMIT,
    "selection_metric": "validation_macro_f1",
    "selected_checkpoint": selected_name,
    "calibration": calibration_summary,
    "threshold_rule": threshold_rule,
    "test": test_summary,
    "test_set_used_for_tuning": False,
}
(output_dir / "evaluation_summary.json").write_text(
    json.dumps(evaluation_summary, indent=2)
)

# Recreate key plots directly into the artifact folder.
fig, axis = plt.subplots(figsize=(14, 12))
sns.heatmap(normalized_confusion, cmap="Blues", ax=axis, xticklabels=classes,
            yticklabels=classes, vmin=0, vmax=1)
axis.set_title("Held-out test confusion matrix — row normalized")
axis.set_xlabel("Predicted")
axis.set_ylabel("Actual")
axis.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.savefig(output_dir / "confusion_matrix_normalized.png", dpi=180)
plt.close(fig)

fig, axis = plt.subplots(figsize=(7, 5))
axis.plot(confidence_policy["coverage"], confidence_policy["accepted_accuracy"])
axis.scatter([threshold_row["coverage"]], [threshold_row["accepted_accuracy"]], color="red")
axis.set_xlabel("Coverage")
axis.set_ylabel("Accepted-case accuracy")
axis.set_title("Validation confidence policy")
axis.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(output_dir / "confidence_policy.png", dpi=180)
plt.close(fig)

archive = shutil.make_archive(
    "/content/cropguard-evaluation-results", "zip", output_dir
)
print(f"Created {archive} ({Path(archive).stat().st_size / 1_000_000:.1f} MB)")
print("SHA-256:", hashlib.sha256(Path(archive).read_bytes()).hexdigest())

In [ ]:
from google.colab import files
files.download("/content/cropguard-evaluation-results.zip")

## Next step

Provide `cropguard-evaluation-results.zip` for review. Do not retrain or adjust the threshold based on held-out test outcomes. The next phase exports and verifies the calibrated INT8 TensorFlow Lite model.